In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder exists:", RAW_DIR.exists())
print("Clean data folder:", CLEAN_DIR)

Project root: C:\Users\Priyanshu\Desktop\hcl\Project
Raw data folder exists: True
Clean data folder: C:\Users\Priyanshu\Desktop\hcl\Project\data\clean


In [2]:
quality_log = []

def log_issue(issue, rule_applied, rows_affected):
    quality_log.append({
        "issue": issue,
        "rule_applied": rule_applied,
        "rows_affected": int(rows_affected)
    })

def normalize_text(series):
    # Convert to pandas string type, trim outer spaces, and collapse repeated spaces.
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\\s+", " ", regex=True)
    )

def read_raw_csv(filename):
    file_path = RAW_DIR / filename
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")
    return pd.read_csv(file_path)

In [3]:
customers_raw = read_raw_csv("customers.csv")
products_raw = read_raw_csv("products.csv")
transactions_raw = read_raw_csv("transactions.csv")

for name, frame in {
    "Customers": customers_raw,
    "Products": products_raw,
    "Transactions": transactions_raw
}.items():
    print(f"\\n{name}: {frame.shape[0]:,} rows x {frame.shape[1]} columns")
    display(frame.head())
    display(frame.info())

\nCustomers: 500 rows x 5 columns


,customer_id,signup_date,city,state,acquisition_channel
0,C00001,2023-12-12,Mumbai,Maharashtra,Referral
1,C00002,2024-05-30,Pune,Maharashtra,Direct
2,C00003,2024-01-19,Mumbai,Maharashtra,Social Media
3,C00004,2024-04-13,Mumbai,Maharashtra,Direct
4,C00005,2023-12-18,Kolkata,West Bengal,Referral


<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   customer_id          500 non-null    str  
 1   signup_date          500 non-null    str  
 2   city                 491 non-null    str  
 3   state                500 non-null    str  
 4   acquisition_channel  494 non-null    str  
dtypes: str(5)
memory usage: 19.7 KB


None

\nProducts: 80 rows x 4 columns


,product_id,product_name,category,unit_price
0,P0001,Coffee Pack 01,Grocery,1815.65
1,P0002,USB Hub 02,Electronics,1128.07
2,P0003,USB Hub 03,Electronics,2287.50
3,P0004,Programming Book 04,Books,5372.23
4,P0005,Novel 05,Books,190.96


... (2 outputs left out)

\nTransactions: 10,291 rows x 9 columns


,invoice_id,customer_id,invoice_date,product_id,quantity,unit_price,discount_pct,payment_method,order_status
0,INV100001,C00001,2025-08-12 10:50:00,P0033,3,5292.76,0.0,Net Banking,Completed
1,INV100001,C00001,2025-08-12 14:11:00,P0075,1,8690.09,0.0,Credit Card,Completed
2,INV100002,C00001,2025-08-17 12:14:00,P0056,3,3458.24,0.1,Credit Card,Completed
3,INV100002,C00001,2025-08-17 20:00:00,P0069,1,597.07,0.0,Net Banking,Completed
4,INV100002,C00001,2025-08-17 14:31:00,P0011,2,7331.66,0.1,Credit Card,Completed


<class 'pandas.DataFrame'>
RangeIndex: 10291 entries, 0 to 10290
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   invoice_id      10291 non-null  str    
 1   customer_id     10209 non-null  str    
 2   invoice_date    10250 non-null  str    
 3   product_id      10291 non-null  str    
 4   quantity        10291 non-null  int64  
 5   unit_price      10250 non-null  float64
 6   discount_pct    10291 non-null  float64
 7   payment_method  10230 non-null  str    
 8   order_status    10291 non-null  str    
dtypes: float64(2), int64(1), str(6)
memory usage: 723.7 KB


None

In [5]:
customers = customers_raw.copy()

# Remove exact duplicate customer records, retaining the first copy.
duplicate_mask = customers.duplicated(keep="first")
duplicate_count = int(duplicate_mask.sum())
customers = customers.loc[~duplicate_mask].copy()
log_issue(
    "Duplicate customer rows",
    "Removed exact duplicate customer rows; retained the first occurrence.",
    duplicate_count
)

# Quarantine customer rows with missing or blank customer IDs.
customers["customer_id"] = normalize_text(customers["customer_id"])
missing_id_mask = customers["customer_id"].isna() | customers["customer_id"].eq("")
missing_id_count = int(missing_id_mask.sum())

if missing_id_count:
    customers.loc[missing_id_mask].to_csv(
        CLEAN_DIR / "customers_quarantine.csv", index=False
    )
    customers = customers.loc[~missing_id_mask].copy()

log_issue(
    "Missing customer_id",
    "Quarantined customer records without a customer identifier.",
    missing_id_count
)

# Fill missing descriptive values with Unknown.
for column in ["city", "state", "acquisition_channel"]:
    customers[column] = normalize_text(customers[column])
    missing_mask = customers[column].isna() | customers[column].eq("")
    count = int(missing_mask.sum())
    customers.loc[missing_mask, column] = "Unknown"
    log_issue(
        f"Missing customer {column}",
        f"Replaced missing or blank {column} values with 'Unknown'.",
        count
    )

# Standardize text to title case.
for column in ["city", "state", "acquisition_channel"]:
    before = customers[column].copy()
    customers[column] = customers[column].str.title()
    changed = int((before.fillna("") != customers[column].fillna("")).sum())
    log_issue(
        f"Inconsistent customer text: {column}",
        f"Trimmed whitespace and standardized {column} to title case.",
        changed
    )

# Parse signup dates. Invalid dates are retained as null because signup date is not
# needed for RFM calculations; the issue is logged for transparency.
customers["signup_date"] = pd.to_datetime(
    customers["signup_date"], errors="coerce"
)
invalid_signup_dates = int(customers["signup_date"].isna().sum())
log_issue(
    "Invalid or missing signup_date",
    "Parsed signup_date; retained customer and left invalid dates null because signup date is not required for RFM.",
    invalid_signup_dates
)
customers["signup_date"] = customers["signup_date"].dt.strftime("%Y-%m-%d")

print("Clean customers:", customers.shape)
display(customers.head())

Clean customers: (500, 5)


,customer_id,signup_date,city,state,acquisition_channel
0,C00001,2023-12-12,Mumbai,Maharashtra,Referral
1,C00002,2024-05-30,Pune,Maharashtra,Direct
2,C00003,2024-01-19,Mumbai,Maharashtra,Social Media
3,C00004,2024-04-13,Mumbai,Maharashtra,Direct
4,C00005,2023-12-18,Kolkata,West Bengal,Referral


In [6]:
products = products_raw.copy()

duplicate_mask = products.duplicated(keep="first")
duplicate_count = int(duplicate_mask.sum())
products = products.loc[~duplicate_mask].copy()
log_issue(
    "Duplicate product rows",
    "Removed exact duplicate product rows; retained the first occurrence.",
    duplicate_count
)

for column in ["product_id", "product_name", "category"]:
    products[column] = normalize_text(products[column])

before_category = products["category"].copy()
products["category"] = products["category"].str.title()
category_changed = int(
    (before_category.fillna("") != products["category"].fillna("")).sum()
)
log_issue(
    "Inconsistent product category text",
    "Trimmed whitespace and standardized category labels to title case.",
    category_changed
)

products["unit_price"] = pd.to_numeric(products["unit_price"], errors="coerce")
invalid_price_mask = products["unit_price"].isna() | (products["unit_price"] <= 0)
invalid_price_count = int(invalid_price_mask.sum())

if invalid_price_count:
    products.loc[invalid_price_mask].to_csv(
        CLEAN_DIR / "products_quarantine.csv", index=False
    )
    products = products.loc[~invalid_price_mask].copy()

log_issue(
    "Invalid product master unit_price",
    "Quarantined products with missing, non-numeric, zero, or negative standard prices.",
    invalid_price_count
)

print("Clean products:", products.shape)
display(products.head())

Clean products: (80, 4)


,product_id,product_name,category,unit_price
0,P0001,Coffee Pack 01,Grocery,1815.65
1,P0002,USB Hub 02,Electronics,1128.07
2,P0003,USB Hub 03,Electronics,2287.50
3,P0004,Programming Book 04,Books,5372.23
4,P0005,Novel 05,Books,190.96


In [7]:
transactions = transactions_raw.copy()

# Preserve duplicate copies for the quarantine/audit file.
duplicate_mask = transactions.duplicated(keep="first")
duplicate_rows = transactions.loc[duplicate_mask].copy()
duplicate_count = int(duplicate_mask.sum())
transactions = transactions.loc[~duplicate_mask].copy()

log_issue(
    "Exact duplicate transaction rows",
    "Quarantined repeated full rows and retained the first occurrence.",
    duplicate_count
)

for column in [
    "invoice_id", "customer_id", "product_id",
    "payment_method", "order_status"
]:
    transactions[column] = normalize_text(transactions[column])

# Normalize known payment labels to a canonical spelling.
before_payment = transactions["payment_method"].copy()
payment_key = (
    transactions["payment_method"]
    .str.casefold()
    .str.replace(r"\\s+", " ", regex=True)
)
payment_map = {
    "upi": "UPI",
    "credit card": "Credit Card",
    "debit card": "Debit Card",
    "net banking": "Net Banking",
    "cash on delivery": "Cash on Delivery"
}
transactions["payment_method"] = payment_key.map(payment_map).fillna(
    transactions["payment_method"]
)
payment_changed = int(
    (before_payment.fillna("") != transactions["payment_method"].fillna("")).sum()
)
log_issue(
    "Inconsistent payment_method text",
    "Trimmed whitespace and mapped known payment labels to canonical values.",
    payment_changed
)

before_status = transactions["order_status"].copy()
transactions["order_status"] = transactions["order_status"].str.title()
status_changed = int(
    (before_status.fillna("") != transactions["order_status"].fillna("")).sum()
)
log_issue(
    "Inconsistent order_status text",
    "Trimmed whitespace and standardized order_status to title case.",
    status_changed
)

transactions["invoice_date"] = pd.to_datetime(
    transactions["invoice_date"], errors="coerce"
)
transactions["quantity"] = pd.to_numeric(
    transactions["quantity"], errors="coerce"
)
transactions["unit_price"] = pd.to_numeric(
    transactions["unit_price"], errors="coerce"
)
transactions["discount_pct"] = pd.to_numeric(
    transactions["discount_pct"], errors="coerce"
)

print("Transaction rows after exact duplicate removal:", len(transactions))
display(transactions.head())

Transaction rows after exact duplicate removal: 10198


,invoice_id,customer_id,invoice_date,product_id,quantity,unit_price,discount_pct,payment_method,order_status
0,INV100001,C00001,2025-08-12 10:50:00,P0033,3,5292.76,0.0,Net Banking,Completed
1,INV100001,C00001,2025-08-12 14:11:00,P0075,1,8690.09,0.0,Credit Card,Completed
2,INV100002,C00001,2025-08-17 12:14:00,P0056,3,3458.24,0.1,Credit Card,Completed
3,INV100002,C00001,2025-08-17 20:00:00,P0069,1,597.07,0.0,Net Banking,Completed
4,INV100002,C00001,2025-08-17 14:31:00,P0011,2,7331.66,0.1,Credit Card,Completed


In [8]:
reason_masks = {}

def add_reason(issue, mask, rule):
    mask = mask.fillna(False)
    reason_masks[issue] = mask
    log_issue(issue, rule, int(mask.sum()))

# Required identifiers and fields.
add_reason(
    "Missing customer_id",
    transactions["customer_id"].isna() | transactions["customer_id"].eq(""),
    "Quarantined transaction lines without a customer ID because RFM cannot attribute them."
)
add_reason(
    "Missing invoice_date",
    transactions["invoice_date"].isna(),
    "Quarantined transaction lines with missing or unparseable invoice dates."
)
add_reason(
    "Missing unit_price",
    transactions["unit_price"].isna(),
    "Quarantined transaction lines with missing or non-numeric unit prices."
)
add_reason(
    "Missing invoice_id",
    transactions["invoice_id"].isna() | transactions["invoice_id"].eq(""),
    "Quarantined transaction lines without an invoice ID."
)
add_reason(
    "Missing product_id",
    transactions["product_id"].isna() | transactions["product_id"].eq(""),
    "Quarantined transaction lines without a product ID."
)

# Invalid numeric values and dates.
add_reason(
    "Zero quantity",
    transactions["quantity"].eq(0),
    "Quarantined zero-quantity lines because they do not represent a purchase or return."
)
add_reason(
    "Negative quantity on non-return order",
    (transactions["quantity"] < 0) & (~transactions["order_status"].eq("Returned")),
    "Quarantined negative quantities unless the order status is Returned."
)
add_reason(
    "Negative unit_price",
    transactions["unit_price"] < 0,
    "Quarantined transaction lines with negative unit prices."
)
add_reason(
    "Invalid discount_pct",
    transactions["discount_pct"].isna()
    | (transactions["discount_pct"] < 0)
    | (transactions["discount_pct"] > 1),
    "Quarantined missing or out-of-range discounts; valid range is 0 through 1."
)
add_reason(
    "Future invoice_date",
    transactions["invoice_date"] > pd.Timestamp("2026-10-31 23:59:59"),
    "Quarantined invoice dates later than the declared data end date of 2026-10-31."
)

# Foreign-key checks.
valid_customer_ids = set(customers["customer_id"].astype(str))
valid_product_ids = set(products["product_id"].astype(str))

add_reason(
    "Unknown customer_id reference",
    transactions["customer_id"].notna()
    & ~transactions["customer_id"].isin(valid_customer_ids),
    "Quarantined transaction lines whose customer ID is absent from customers.csv."
)
add_reason(
    "Unknown product_id reference",
    transactions["product_id"].notna()
    & ~transactions["product_id"].isin(valid_product_ids),
    "Quarantined transaction lines whose product ID is absent from products.csv."
)

# Combine reasons so each rejected source row is quarantined only once.
any_invalid = pd.Series(False, index=transactions.index)
for mask in reason_masks.values():
    any_invalid |= mask

rejected = transactions.loc[any_invalid].copy()
if not rejected.empty:
    rejected["quarantine_reason"] = [
        "; ".join(
            issue for issue, mask in reason_masks.items() if bool(mask.loc[index])
        )
        for index in rejected.index
    ]

# Keep duplicate copies and their reason in the audit file.
if not duplicate_rows.empty:
    duplicate_rows["quarantine_reason"] = (
        "Exact duplicate of an earlier transaction row"
    )

transactions_quarantine = pd.concat(
    [rejected, duplicate_rows], ignore_index=True, sort=False
)
transactions_quarantine.to_csv(
    CLEAN_DIR / "transactions_quarantine.csv", index=False
)

transactions_clean = transactions.loc[~any_invalid].copy()

print("Rejected rows for one or more validation issues:", len(rejected))
print("Duplicate copies quarantined:", len(duplicate_rows))
print("Total quarantine records:", len(transactions_quarantine))
print("Retained transaction rows:", len(transactions_clean))

Rejected rows for one or more validation issues: 220
Duplicate copies quarantined: 93
Total quarantine records: 313
Retained transaction rows: 9978


In [9]:
reason_masks = {}

def add_reason(issue, mask, rule):
    mask = mask.fillna(False)
    reason_masks[issue] = mask
    log_issue(issue, rule, int(mask.sum()))

# Required identifiers and fields.
add_reason(
    "Missing customer_id",
    transactions["customer_id"].isna() | transactions["customer_id"].eq(""),
    "Quarantined transaction lines without a customer ID because RFM cannot attribute them."
)
add_reason(
    "Missing invoice_date",
    transactions["invoice_date"].isna(),
    "Quarantined transaction lines with missing or unparseable invoice dates."
)
add_reason(
    "Missing unit_price",
    transactions["unit_price"].isna(),
    "Quarantined transaction lines with missing or non-numeric unit prices."
)
add_reason(
    "Missing invoice_id",
    transactions["invoice_id"].isna() | transactions["invoice_id"].eq(""),
    "Quarantined transaction lines without an invoice ID."
)
add_reason(
    "Missing product_id",
    transactions["product_id"].isna() | transactions["product_id"].eq(""),
    "Quarantined transaction lines without a product ID."
)

# Invalid numeric values and dates.
add_reason(
    "Zero quantity",
    transactions["quantity"].eq(0),
    "Quarantined zero-quantity lines because they do not represent a purchase or return."
)
add_reason(
    "Negative quantity on non-return order",
    (transactions["quantity"] < 0) & (~transactions["order_status"].eq("Returned")),
    "Quarantined negative quantities unless the order status is Returned."
)
add_reason(
    "Negative unit_price",
    transactions["unit_price"] < 0,
    "Quarantined transaction lines with negative unit prices."
)
add_reason(
    "Invalid discount_pct",
    transactions["discount_pct"].isna()
    | (transactions["discount_pct"] < 0)
    | (transactions["discount_pct"] > 1),
    "Quarantined missing or out-of-range discounts; valid range is 0 through 1."
)
add_reason(
    "Future invoice_date",
    transactions["invoice_date"] > pd.Timestamp("2026-10-31 23:59:59"),
    "Quarantined invoice dates later than the declared data end date of 2026-10-31."
)

# Foreign-key checks.
valid_customer_ids = set(customers["customer_id"].astype(str))
valid_product_ids = set(products["product_id"].astype(str))

add_reason(
    "Unknown customer_id reference",
    transactions["customer_id"].notna()
    & ~transactions["customer_id"].isin(valid_customer_ids),
    "Quarantined transaction lines whose customer ID is absent from customers.csv."
)
add_reason(
    "Unknown product_id reference",
    transactions["product_id"].notna()
    & ~transactions["product_id"].isin(valid_product_ids),
    "Quarantined transaction lines whose product ID is absent from products.csv."
)

# Combine reasons so each rejected source row is quarantined only once.
any_invalid = pd.Series(False, index=transactions.index)
for mask in reason_masks.values():
    any_invalid |= mask

rejected = transactions.loc[any_invalid].copy()
if not rejected.empty:
    rejected["quarantine_reason"] = [
        "; ".join(
            issue for issue, mask in reason_masks.items() if bool(mask.loc[index])
        )
        for index in rejected.index
    ]

# Keep duplicate copies and their reason in the audit file.
if not duplicate_rows.empty:
    duplicate_rows["quarantine_reason"] = (
        "Exact duplicate of an earlier transaction row"
    )

transactions_quarantine = pd.concat(
    [rejected, duplicate_rows], ignore_index=True, sort=False
)
transactions_quarantine.to_csv(
    CLEAN_DIR / "transactions_quarantine.csv", index=False
)

transactions_clean = transactions.loc[~any_invalid].copy()

print("Rejected rows for one or more validation issues:", len(rejected))
print("Duplicate copies quarantined:", len(duplicate_rows))
print("Total quarantine records:", len(transactions_quarantine))
print("Retained transaction rows:", len(transactions_clean))

Rejected rows for one or more validation issues: 220
Duplicate copies quarantined: 93
Total quarantine records: 313
Retained transaction rows: 9978


In [10]:
missing_payment_mask = (
    transactions_clean["payment_method"].isna()
    | transactions_clean["payment_method"].eq("")
)
missing_payment_count = int(missing_payment_mask.sum())
transactions_clean.loc[missing_payment_mask, "payment_method"] = "Unknown"

log_issue(
    "Missing payment_method",
    "Replaced missing or blank payment methods in retained rows with 'Unknown'.",
    missing_payment_count
)

allowed_statuses = {"Completed", "Cancelled", "Returned"}
unknown_status_mask = ~transactions_clean["order_status"].isin(allowed_statuses)
unknown_status_count = int(unknown_status_mask.sum())

log_issue(
    "Unrecognized order_status",
    "Retained unrecognized statuses for review; exclude them from completed-purchase RFM.",
    unknown_status_count
)

# Export dates and numeric columns in consistent formats.
transactions_clean["invoice_date"] = (
    transactions_clean["invoice_date"].dt.strftime("%Y-%m-%d %H:%M:%S")
)
transactions_clean["quantity"] = transactions_clean["quantity"].astype("int64")
transactions_clean["unit_price"] = transactions_clean["unit_price"].round(2)
transactions_clean["discount_pct"] = transactions_clean["discount_pct"].round(2)

display(transactions_clean.head())

,invoice_id,customer_id,invoice_date,product_id,quantity,unit_price,discount_pct,payment_method,order_status
0,INV100001,C00001,2025-08-12 10:50:00,P0033,3,5292.76,0.0,Net Banking,Completed
1,INV100001,C00001,2025-08-12 14:11:00,P0075,1,8690.09,0.0,Credit Card,Completed
2,INV100002,C00001,2025-08-17 12:14:00,P0056,3,3458.24,0.1,Credit Card,Completed
3,INV100002,C00001,2025-08-17 20:00:00,P0069,1,597.07,0.0,Net Banking,Completed
4,INV100002,C00001,2025-08-17 14:31:00,P0011,2,7331.66,0.1,Credit Card,Completed


In [11]:
customers.to_csv(CLEAN_DIR / "customers.csv", index=False)
products.to_csv(CLEAN_DIR / "products.csv", index=False)
transactions_clean.to_csv(CLEAN_DIR / "transactions.csv", index=False)
transactions_quarantine.to_csv(
    CLEAN_DIR / "transactions_quarantine.csv", index=False
)

quality_log_df = pd.DataFrame(
    quality_log,
    columns=["issue", "rule_applied", "rows_affected"]
)
quality_log_df.to_csv(CLEAN_DIR / "data_quality_log.csv", index=False)

print("Saved cleaned files to:", CLEAN_DIR)
display(quality_log_df)

Saved cleaned files to: C:\Users\Priyanshu\Desktop\hcl\Project\data\clean


,issue,rule_applied,rows_affected
0,Duplicate customer rows,Removed exact duplicate customer rows; retaine...,0
1,Missing customer_id,Quarantined customer records without a custome...,0
2,Missing customer city,Replaced missing or blank city values with 'Un...,9
3,Missing customer state,Replaced missing or blank state values with 'U...,0
4,Missing customer acquisition_channel,Replaced missing or blank acquisition_channel ...,6
5,Inconsistent customer text: city,Trimmed whitespace and standardized city to ti...,0
6,Inconsistent customer text: state,Trimmed whitespace and standardized state to t...,0
7,Inconsistent customer text: acquisition_channel,Trimmed whitespace and standardized acquisitio...,0
8,Invalid or missing signup_date,Parsed signup_date; retained customer and left...,0
9,Duplicate customer rows,Removed exact duplicate customer rows; retaine...,0


In [13]:
check_results = {
    "No exact duplicate transaction rows": not transactions_clean.duplicated().any(),
    "No missing invoice_id": transactions_clean["invoice_id"].notna().all(),
    "No missing customer_id": transactions_clean["customer_id"].notna().all(),
    "No missing product_id": transactions_clean["product_id"].notna().all(),
    "All retained unit prices are positive": (transactions_clean["unit_price"] > 0).all(),
    "No retained zero quantities": (transactions_clean["quantity"] != 0).all(),
    "No future invoice dates": (
        pd.to_datetime(transactions_clean["invoice_date"])
        <= pd.Timestamp("2026-10-31 23:59:59")
    ).all(),
    "All customer references match": transactions_clean["customer_id"].isin(
        customers["customer_id"]
    ).all(),
    "All product references match": transactions_clean["product_id"].isin(
        products["product_id"]
    ).all()
}

validation = pd.DataFrame(
    [{"check": name, "result": "PASS" if passed else "FAIL"}
     for name, passed in check_results.items()]
)
display(validation)

print("\nFinal row counts")
print("Customers:", len(customers))
print("Products:", len(products))
print("Transactions:", len(transactions_clean))
print("Quarantine:", len(transactions_quarantine))

,check,result
0,No exact duplicate transaction rows,PASS
1,No missing invoice_id,PASS
2,No missing customer_id,PASS
3,No missing product_id,PASS
4,All retained unit prices are positive,PASS
5,No retained zero quantities,PASS
6,No future invoice dates,PASS
7,All customer references match,PASS
8,All product references match,PASS



Final row counts
Customers: 500
Products: 80
Transactions: 9978
Quarantine: 313
